In [1]:
# importing the required libraries
import torch
print(torch.__version__)

2.11.0+cpu


In [8]:
# import the required libraries
import os
import glob
import requests
import numpy as np
from IPython.display import Audio, display


os.makedirs("data/pdfs", exist_ok=True)

LLM_MODEL = "llama3"
WHISPER_SIZE = "small"

LAT, LON = 20.9, 79.0

print("✅ Configuration loaded")
print("LLM:", LLM_MODEL)
print("Whisper:", WHISPER_SIZE)
print("Location:", LAT, LON)

✅ Configuration loaded
LLM: llama3
Whisper: small
Location: 20.9 79.0


In [10]:
# Agri Voice - Step 3
# Knowledge Base + RAG

import glob
import chromadb
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")

emb = SentenceTransformer("intfloat/multilingual-e5-small")

# Persistent ChromaDB database
client = chromadb.PersistentClient(path="db")

col = client.get_or_create_collection(
    name="agri"
)

print("✅ Embedding model loaded")
print("✅ ChromaDB collection ready")


def ingest():

    docs = []

    # Read agricultural PDF files
    for f in glob.glob("data/pdfs/*.pdf"):

        try:
            reader = PdfReader(f)

            text = " ".join(
                page.extract_text() or ""
                for page in reader.pages
            )

            text = " ".join(text.split())

            # Create chunks
            for i in range(0, len(text), 1300):

                chunk = text[i:i + 1500]

                if chunk.strip():

                    docs.append(
                        (
                            f"{os.path.basename(f)}-{i}",
                            chunk
                        )
                    )

        except Exception as e:
            print(f"⚠️ Error reading {f}: {e}")


    # If no PDF is available, use sample agricultural knowledge
    if not docs:

        docs = [

            (
                "sample_cotton",
                """Cotton: sowing is generally done during June-July.
                Common spacing can be around 90 x 60 cm depending on
                variety and local agricultural recommendations.
                Pink bollworm can affect cotton. Pheromone traps,
                field monitoring and recommended pest management
                practices can help control it."""
            ),

            (
                "sample_soybean",
                """Soybean is generally sown with the onset of monsoon.
                Seed rate depends on variety and seed size, commonly
                around 65-75 kg per hectare.
                Yellow mosaic virus is transmitted by whiteflies.
                Farmers should monitor the crop and follow recommended
                vector management practices."""
            ),

            (
                "sample_tur",
                """Tur or pigeon pea is generally sown during June-July.
                Wilt is a soil-borne disease.
                Resistant varieties, healthy seed and recommended
                Trichoderma seed treatment can help in disease management."""
            )
        ]


    ids = [d[0] for d in docs]
    documents = [d[1] for d in docs]

    # E5 models use "passage:" for documents
    embeddings = emb.encode(
        ["passage: " + d for d in documents],
        normalize_embeddings=True
    ).tolist()


    col.upsert(
        ids=ids,
        documents=documents,
        embeddings=embeddings
    )

    print(f"✅ Indexed {len(docs)} agricultural chunks")


ingest()

ModuleNotFoundError: No module named 'chromadb'